In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

plt.style.use('dark_background')
sns.set_palette('viridis')
print("🌽 Environment ready for Kenya Maize Market Intelligence ML Pipeline.")

#load data and feature engineering
np.random.seed(42)
dates = pd.date_range(start="2024-01-01",end="2026-02-01",freq="MS")
n_periods = len(dates)

df = pd.DataFrame({
    'price': 4200 + np.cumsum(np.random.randn(n_periods) * 40),
    'rainfall_mm': 80 + np.sin(np.linspace(0, 3 * np.pi, n_periods)) * 30,
}, index=dates)


df['lag_1_price'] = df['price'].shift(1)
df['lag_12_price'] = df['price'].shift(12)
df['rolling_3m_mean'] = df['price'].shift(1).rolling(window=3).mean()
df = df.dropna()

print(f"Dataset shape after feature engineering:{df.shape}")
df.head()


#chronological train/test split
train_size = int(len(df) * 0.8)
train,test = df.iloc[:train_size],df.iloc[train_size:]


features = ['rainfall_mm','lag_1_price','lag_12_price','rolling_3m_mean']
x_train,y_train = train[features],train['price']
x_test,y_test = test[features],test['price']
print(f"Training samples: {len(x_train)} | Testing samples: {len(x_test)}")

#baseline models implementation

results = {}

def evaluate_and_store(model_name,y_true,y_pred):
    rmse = np.sqrt(mean_squared_error(y_true,y_pred))
    mae = mean_absolute_error(y_true,y_pred)
    results[model_name] = {'RMSE':rmse,'MAE':mae}
    print(f"[{model_name}] RMSE:{rmse:.2f} | MAE:{mae:.2f}")


naive_pred = test['price'].shift(1).fillna(train['price'].iloc[-1])
evaluate_and_store("Baseline 1: Naive",y_test,naive_pred)


seasonal_naive_pred = df['price'].shift(12).loc[test.index]
evaluate_and_store("Baseline 2: Seasonal Naive",y_test,seasonal_naive_pred)

historical_median_pred = np.full_like(y_test,train['price'].median())
evaluate_and_store("Baseline 3: Historical Median", y_test, historical_median_pred)

#machine learning model training and evaluation
models = {
    "Model 1: Ridge Regression": Ridge(alpha=1.0),
    "Model 2: Random Forest": RandomForestRegressor(n_estimators=100,random_state=42),
     "Model 3: XGBoost":XGBRegressor(n_estimators=100,learning_rate=0.05,random_state=42),
     "Model 4:lightGBM":LGBMRegressor(n_estimators=100,learning_rate=0.05,random_state=42,verbose=-1))
}

trained_models = {}
predictions = {}

for name,model in models.items():
    model.fit(x_train,y_train)
    preds = model.predict(x_test)
    trained_models[name] = model
    predictions[name] = preds
    



